# Phase 3 — Dynamic and Timbral Feature Extraction



RQ2
How do dynamic accents and timbral brightness interact across different performance eras, and can this relationship mathematically quantify Herbie Hancock's expressive touch?

In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

files = ['chameleon.wav', 'cantaloupe.wav', 'rockit.wav']
titles = {
    'chameleon.wav': 'Chameleon (1970s)',
    'cantaloupe.wav': 'Cantaloupe Island (1960s)',
    'rockit.wav': 'Rockit (1983)'
}

hop_length = 512
frame_length = 2048
tolerance_ms = 80.0

audio_data = {}
sr_data = {}
onset_times_data = {}
beat_times_data = {}


for file in files:
    path = f'../data/audio/{file}'
    y, sr = librosa.load(path, sr=None)
    audio_data[file] = y
    sr_data[file] = sr
    
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, hop_length=hop_length, backtrack=True)
    onset_times_data[file] = librosa.frames_to_time(onset_frames, sr=sr, hop_length=hop_length)
    
    tempo, beat_frames = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr, hop_length=hop_length)
    beat_times_data[file] = librosa.frames_to_time(beat_frames, sr=sr, hop_length=hop_length)
    
    print(f"Loaded: {titles[file]} | Onsets detected: {len(onset_frames)}")

c:\Users\grasd\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: Chameleon (1970s) | Onsets detected: 414
Loaded: Cantaloupe Island (1960s) | Onsets detected: 257
Loaded: Rockit (1983) | Onsets detected: 122


In [2]:
test_file = 'chameleon.wav'
y_test = audio_data[test_file]
sr_test = sr_data[test_file]

rms_test = librosa.feature.rms(y=y_test, frame_length=frame_length, hop_length=hop_length)[0]
centroid_test = librosa.feature.spectral_centroid(y=y_test, sr=sr_test, n_fft=frame_length, hop_length=hop_length)[0]

print(f"RMS shape: {rms_test.shape} | Min: {rms_test.min():.4f}, Max: {rms_test.max():.4f}")
print(f"Centroid shape: {centroid_test.shape} | Min: {centroid_test.min():.1f} Hz, Max: {centroid_test.max():.1f} Hz")

test_onsets = librosa.onset.onset_detect(
    onset_envelope=librosa.onset.onset_strength(y=y_test, sr=sr_test, hop_length=hop_length),
    sr=sr_test, 
    hop_length=hop_length, 
    backtrack=True
)

print("\nFirst 5 onset frames sample:")
for frame in test_onsets[:5]:
    print(f"Frame {frame:4d} -> RMS: {rms_test[frame]:.4f} | Spectral Centroid: {centroid_test[frame]:.1f} Hz")

RMS shape: (6202,) | Min: 0.0000, Max: 0.4738
Centroid shape: (6202,) | Min: 0.0 Hz, Max: 7535.8 Hz

First 5 onset frames sample:
Frame    6 -> RMS: 0.2629 | Spectral Centroid: 1553.8 Hz
Frame   15 -> RMS: 0.0664 | Spectral Centroid: 895.1 Hz
Frame   31 -> RMS: 0.1171 | Spectral Centroid: 4728.5 Hz
Frame   42 -> RMS: 0.2904 | Spectral Centroid: 2629.3 Hz
Frame   63 -> RMS: 0.1002 | Spectral Centroid: 1878.0 Hz
